In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Đọc dữ liệu
file_path = '/content/Banking_Transactional_Dataset.xlsx'
df = pd.read_excel(file_path)

# 2. Feature Engineering (Khai phá đặc trưng thời gian)
df['Year'] = df['TransactionDate'].dt.year
df['Month'] = df['TransactionDate'].dt.month
df['DayOfWeek'] = df['TransactionDate'].dt.dayofweek

# 3. Feature Selection (Loại bỏ các cột không có giá trị dự báo)
cols_to_drop = ['TransactionID', 'CustomerID', 'TransactionDate', 'BranchLat', 'BranchLong', 'Currency']
df_ml = df.drop(columns=cols_to_drop)

print("Kích thước dữ liệu sau khi loại bỏ cột thừa:", df_ml.shape)

Kích thước dữ liệu sau khi loại bỏ cột thừa: (20000, 16)


In [2]:
# 1. Tách biến mục tiêu (y) là số tiền giao dịch và các biến đầu vào (X)
X = df_ml.drop(columns=['Amount'])
y = df_ml['Amount']

# 2. Encoding Categorical Variables (Mã hóa One-Hot cho các cột chữ)
X_encoded = pd.get_dummies(X, drop_first=True)

# 3. Train/Test Split (Chia tập dữ liệu 80% Train - 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 4. Data Scaling (Chuẩn hóa thang đo dữ liệu)
scaler = StandardScaler()
# Lưu ý: Chỉ fit trên tập Train để tránh rò rỉ dữ liệu (Data Leakage)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Kích thước tập Train:", X_train_scaled.shape)
print("Kích thước tập Test:", X_test_scaled.shape)

Kích thước tập Train: (16000, 39)
Kích thước tập Test: (4000, 39)


In [3]:
# Khởi tạo 3 mô hình cơ bản
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42, objective='reg:squarederror')
}

results = []
print("Đang huấn luyện và đánh giá các mô hình cơ bản...\n")

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    results.append({
        "Mô hình": name,
        "MAE": round(mean_absolute_error(y_test, y_pred), 2),
        "RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred)), 2),
        "R² Score": round(r2_score(y_test, y_pred), 4)
    })

results_df = pd.DataFrame(results)
display(results_df)

Đang huấn luyện và đánh giá các mô hình cơ bản...



,Mô hình,MAE,RMSE,R² Score
0,Linear Regression,2535.24,3075.49,0.2472
1,Random Forest,2592.28,3177.67,0.1964
2,XGBoost,2629.17,3253.64,0.1575
